In [1]:
# @airawatraj - Fiduciary Agent: The Risk-Aware Orchestrator
# ==================================================================
# An autonomous Fiduciary Agent powered by a local model (Cogni-Brain)
# that enforces enterprise risk governance (CLV vs Refund) 
# using strict tool-first protocols.
# ==================================================================

# --- STEP 1: SETUP & IMPORTS ---
!pip install openai --upgrade --quiet > /dev/null 2>&1

import os
import json
import logging
import asyncio
from openai import AsyncOpenAI

# Initialize Local Client
try:
    # Pointing to the local DGX Spark server
    client = AsyncOpenAI(
        base_url="http://192.168.20.91:8000/v1",
        api_key="local-key-not-needed" # Local servers usually ignore this
    )
    print("Step 1 Complete: Local environment configured and client secured.")
except Exception as e:
    print(f"Connection Error: {e}")

logging.basicConfig(level=logging.ERROR)
MODEL_NAME = "Cogni-Brain"

Step 1 Complete: Local environment configured and client secured.


In [2]:
# --- STEP 2: THE HANDS (Custom Tools) ---

# Mock Database
DB = {
    "ORD-VIP": {"status": "lost", "value": 150, "clv": 5000, "tier": "VIP"},
    "ORD-RISK": {"status": "lost", "value": 800, "clv": 200, "tier": "New"}
}

def check_order_status(order_id: str):
    """Step 1: Retrieves order details."""
    print(f"   [HANDS] Checking DB for {order_id}...")
    data = DB.get(order_id)
    if data:
        return f"DETAILS: Value=${data['value']}, CLV=${data['clv']}, Status={data['status']}."
    return "Order not found."

def execute_refund(order_id: str):
    """Step 2: Processes refund based on Risk Policy."""
    print(f"   [HANDS] Attempting refund for {order_id}...")
    data = DB.get(order_id)
    
    if not data:
        return "Error: Order not found."
    
    # Enterprise Logic (Hidden from LLM)
    if data['value'] > data['clv'] or data['value'] > 500:
        return f"REFUND DENIED. Risk Alert: Order Value ${data['value']} exceeds Limit/CLV. Escalate to Human."
    
    return f"REFUND SUCCESS. ${data['value']} returned to customer."

# Map for easy execution
available_functions = {
    "check_order_status": check_order_status,
    "execute_refund": execute_refund
}

# Standard OpenAI Tool Schema
tools = [
    {
        "type": "function",
        "function": {
            "name": "check_order_status",
            "description": "Retrieves order details including value and Customer Lifetime Value (CLV).",
            "parameters": {
                "type": "object",
                "properties": {
                    "order_id": {"type": "string", "description": "The ID of the order"}
                },
                "required": ["order_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "execute_refund",
            "description": "Processes a refund for a given order ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "order_id": {"type": "string", "description": "The ID of the order to refund"}
                },
                "required": ["order_id"]
            }
        }
    }
]

print("Step 2 Complete: Custom Tools and Schemas defined.")

Step 2 Complete: Custom Tools and Schemas defined.


In [3]:
# --- STEP 3: THE BRAIN (Strict Logic Chain) ---

SYSTEM_PROMPT = """
You are a Robotic Process Automation (RPA) Controller.

**STRICT EXECUTION PROTOCOL:**
1.  **INPUT:** User provides Order ID.
2.  **STEP 1:** Call `check_order_status(order_id)`.
3.  **OBSERVE:** Read the status.
4.  **STEP 2 (MANDATORY):** Call `execute_refund(order_id)`.
    * You CANNOT claim a refund is done until you call this tool.
    * The tool will decide if it is Approved or Denied.
5.  **OUTPUT:** Report the exact result returned by `execute_refund` to the user.

**FORBIDDEN:**
* Do NOT say "I have processed the refund" before calling Step 2.
* Do NOT make up the refund decision yourself. Rely ONLY on the tool.
"""

print("Step 3 Complete: System Prompt prepared.")

Step 3 Complete: System Prompt prepared.


In [4]:
# --- STEP 4 & 5: THE INFRASTRUCTURE & DEMO ---

class LocalSessionRunner:
    def __init__(self, system_prompt):
        # Initialize memory with the system instruction
        self.messages = [{"role": "system", "content": system_prompt}]
        self.turn_count = 0

    async def chat(self, user_query):
        self.turn_count += 1
        print(f"\n[MEMORY] Active Turn: {self.turn_count}")
        print(f"USER: {user_query}")
        
        self.messages.append({"role": "user", "content": user_query})

        # Step 1: Send query to local model
        response = await client.chat.completions.create(
            model=MODEL_NAME,
            messages=self.messages,
            tools=tools,
            temperature=0.1
        )
        
        response_message = response.choices[0].message
        self.messages.append(response_message)

        # Step 2: Handle Tool Calls if requested by the model
        if response_message.tool_calls:
            for tool_call in response_message.tool_calls:
                function_name = tool_call.function.name
                function_to_call = available_functions[function_name]
                function_args = json.loads(tool_call.function.arguments)
                
                print(f"HAND CALL: {function_name}({function_args})")
                
                # Execute Python function
                function_response = function_to_call(**function_args)
                
                # Append tool result to memory
                self.messages.append({
                    "tool_call_id": tool_call.id,
                    "role": "tool",
                    "name": function_name,
                    "content": function_response,
                })
            
            # Step 3: Get final answer from model after tool execution
            final_response = await client.chat.completions.create(
                model=MODEL_NAME,
                messages=self.messages,
            )
            final_text = final_response.choices[0].message.content
            self.messages.append({"role": "assistant", "content": final_text})
            print(f"BRAIN: {final_text}")
        else:
            # If no tools were called
            print(f"BRAIN: {response_message.content}")

async def run_demo():
    print(f"AGENT LOADED: {MODEL_NAME} at Local DGX Spark")
    print("--------------------------------------------------")
    
    runner = LocalSessionRunner(SYSTEM_PROMPT)
    
    # --- SCENARIO 1: VIP CUSTOMER ---
    await runner.chat("My order ORD-VIP is lost. Please refund it.")
    
    print("-" * 50)
    
    # --- SCENARIO 2: RISKY CUSTOMER ---
    # Note: Memory persists from the previous turn
    await runner.chat("Okay, what about ORD-RISK? Refund that one too.")

# Execute Demo
await run_demo()

AGENT LOADED: Cogni-Brain at Local DGX Spark
--------------------------------------------------

[MEMORY] Active Turn: 1
USER: My order ORD-VIP is lost. Please refund it.
HAND CALL: check_order_status({'order_id': 'ORD-VIP'})
   [HANDS] Checking DB for ORD-VIP...
BRAIN: 

<tool_call>
<function=execute_refund>
<parameter=order_id>
ORD-VIP
</parameter>
</function>
</tool_call>
--------------------------------------------------

[MEMORY] Active Turn: 2
USER: Okay, what about ORD-RISK? Refund that one too.
HAND CALL: check_order_status({'order_id': 'ORD-RISK'})
   [HANDS] Checking DB for ORD-RISK...
BRAIN: 

<tool_call>
<function=execute_refund>
<parameter=order_id>
ORD-RISK
</parameter>
</function>
</tool_call>


In [5]:
# --- STEP 5: EXECUTE DEMO ---
await run_demo()

AGENT LOADED: Cogni-Brain at Local DGX Spark
--------------------------------------------------

[MEMORY] Active Turn: 1
USER: My order ORD-VIP is lost. Please refund it.
HAND CALL: check_order_status({'order_id': 'ORD-VIP'})
   [HANDS] Checking DB for ORD-VIP...
BRAIN: 

<tool_call>
<function=execute_refund>
<parameter=order_id>
ORD-VIP
</parameter>
</function>
</tool_call>
--------------------------------------------------

[MEMORY] Active Turn: 2
USER: Okay, what about ORD-RISK? Refund that one too.
HAND CALL: check_order_status({'order_id': 'ORD-RISK'})
   [HANDS] Checking DB for ORD-RISK...
BRAIN: 

<tool_call>
<function=execute_refund>
<parameter=order_id>
ORD-RISK
</parameter>
</function>
</tool_call>
